### Configuración Inicial y Validaciones de Esquema

In [0]:
from pyspark.sql.functions import col, upper, trim, when
from delta.tables import DeltaTable

# esquema Silver 
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

print("Entorno Silver configurado correctamente.")

### reglas de Calidad y Procesamiento para ventas

In [0]:
# Rutas de tablas
bronze_ventas = "workspace.bronze.ventas"
silver_ventas = "workspace.silver.ventas"

# Leer Bronze
df_ventas_bronze = spark.read.table(bronze_ventas)

# REGLAS DE CALIDAD Y NEGOCIO :
# 1. Deduplicación: Eliminar registros exactamente iguales.
# 2. Nulos Críticos: Eliminar filas donde el 'cliente_id' o el 'producto' sean nulos (huérfanos).
# 3. Regla de Negocio (Nulos en Cantidad): Si la cantidad es nula, la transacción es inválida, se elimina.
# 4. Enriquecimiento: Calcular 'valor_total'.

df_ventas_silver = (
    df_ventas_bronze
    .dropDuplicates()
    .dropna(subset=["cliente_id", "producto", "cantidad", "valor_unitario"]) 
    .withColumn("valor_total", col("cantidad") * col("valor_unitario"))
    # Regla adicional: Filtrar cantidades negativas o en cero que puedan ser errores de sistema
    .filter(col("cantidad") > 0)
)

#  MERGE en Silver
if spark.catalog.tableExists(silver_ventas):
    delta_ventas = DeltaTable.forName(spark, silver_ventas)
    (
        delta_ventas.alias("t")
        .merge(df_ventas_silver.alias("s"), 
               "t.fecha_venta = s.fecha_venta AND t.cliente_id = s.cliente_id AND t.producto = s.producto")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_ventas_silver.write.format("delta").mode("overwrite").saveAsTable(silver_ventas)

print(f"Ventas procesadas y guardadas en {silver_ventas}")

###Reglas de Calidad y Procesamiento para CLIENTES

In [0]:
bronze_clientes_path = "/Volumes/workspace/default/landing/clientes_sql/"
silver_clientes = "workspace.silver.clientes"

df_clientes_bronze = spark.read.parquet(bronze_clientes_path)

# REGLAS DE CALIDAD Y NEGOCIO (CLIENTES):
# 1. Deduplicación por Llave: Un cliente no puede estar dos veces con el mismo ID.
# 2. Estandarización de Strings: Convertir la ciudad a mayúsculas y quitar espacios en blanco (TRIM) para evitar que "Bogota " y "BOGOTA" sean ciudades distintas en Gold.
# 3. Nulos: Eliminar si el ID del cliente es nulo.

df_clientes_silver = (
    df_clientes_bronze
    .dropna(subset=["cliente_id"])
    .dropDuplicates(["cliente_id"]) # Deduplicar estrictamente por el ID
    .withColumn("ciudad", upper(trim(col("ciudad"))))
    .withColumn("nombre", trim(col("nombre")))
)

# Guardar o hacer MERGE en Silver
if spark.catalog.tableExists(silver_clientes):
    delta_clientes = DeltaTable.forName(spark, silver_clientes)
    (
        delta_clientes.alias("t")
        .merge(df_clientes_silver.alias("s"), "t.cliente_id = s.cliente_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_clientes_silver.write.format("delta").mode("overwrite").saveAsTable(silver_clientes)

print(f"Clientes procesados y guardados en {silver_clientes}")

In [0]:
bronze_inventario_path = "/Volumes/workspace/default/landing/inventario_api/"
silver_inventario = "workspace.silver.inventario"

df_inventario_bronze = spark.read.json(bronze_inventario_path)

# REGLAS DE CALIDAD Y NEGOCIO (INVENTARIO):
# 1. Deduplicación: Por producto y almacén.
# 2. Regla de Negocio (Stock): El stock no puede ser negativo. Si viene negativo por error de API, lo forzamos a 0 usando la función 'when'.
# 3. Limpieza: Trim en los nombres de almacén.

df_inventario_silver = (
    df_inventario_bronze
    .dropDuplicates(["producto", "almacen"])
    .dropna(subset=["producto"])
    .withColumn("stock_disponible", when(col("stock_disponible") < 0, 0).otherwise(col("stock_disponible")))
    .withColumn("almacen", trim(col("almacen")))
)

#  MERGE en Silver
if spark.catalog.tableExists(silver_inventario):
    delta_inventario = DeltaTable.forName(spark, silver_inventario)
    (
        delta_inventario.alias("t")
        .merge(df_inventario_silver.alias("s"), "t.producto = s.producto AND t.almacen = s.almacen")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_inventario_silver.write.format("delta").mode("overwrite").saveAsTable(silver_inventario)

print(f" Inventario procesado y guardado en {silver_inventario}")

###Captura de Datos no Complacientes

In [0]:
from pyspark.sql.functions import col, lit, when, concat_ws, current_date

# 1. Definir las rutas de las tablas
bronze_ventas = "workspace.bronze.ventas"
silver_ventas = "workspace.silver.ventas"
quarantine_ventas = "workspace.silver.ventas_quarentena"

# 2. Leer datos desde Bronze
df_bronze = spark.read.table(bronze_ventas)

# 3. Identificar registros que NO cumplen con las reglas
# Definimos las condiciones de falla
condicion_nulos = (col("cliente_id").isNull()) | (col("producto").isNull()) | (col("cantidad").isNull())
condicion_negativos = (col("cantidad") <= 0)

# 4. Crear el DataFrame de Cuarentena con el motivo del rechazo
df_rechazados = df_bronze.filter(condicion_nulos | condicion_negativos).withColumn(
    "motivo_rechazo",
    when(condicion_nulos, "Campos obligatorios nulos")
    .when(condicion_negativos, "Cantidad igual o menor a cero")
    .otherwise("Error de integridad desconocido")
).withColumn("fecha_captura", current_date())

# 5. Guardar en la tabla de Cuarentena Append 
(
    df_rechazados.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(quarantine_ventas)
)

print(f"Se han enviado {df_rechazados.count()} registros a la tabla de cuarentena: {quarantine_ventas}")

# 6. Filtrar los registros LIMPIOS para continuar el flujo normal hacia Silver
df_ventas_silver = df_bronze.filter(~(condicion_nulos | condicion_negativos)).dropDuplicates()